In [ ]:
import pandas as pd

# 1. Chargement des données (tu peux aussi mettre le chemin d'un fichier CSV local si tu l'as téléchargé)
url_data = "https://epoch.ai/data/ai_chip_owners_cumulative_by_chip_type.csv"
print("Chargement des données...")

try:
    df = pd.read_csv(url_data)
except Exception as e:
    print(f"Erreur de téléchargement, vérifie l'URL ou ton fichier local. Détail : {e}")
    exit()

# Afficher les colonnes réelles du fichier pour déboguer
print("\n--- Colonnes disponibles dans le fichier source ---")
print(df.columns.tolist())
print("---------------------------------------------------\n")

# 2. Définition des noms possibles pour chaque colonne clé (pour éviter les erreurs KeyError)
colonnes_possibles = {
    'Proprietaire': ['Owner', 'Company', 'Organization', 'Name'],
    'Modele_Puce': ['Chip type', 'Hardware', 'Accelerator', 'Chip type (primary)'],
    'Quantite': ['Cumulative units', 'Quantity', 'Chip quantity (primary)', 'Amount', 'Units'],
    'Date': ['Quarter', 'Date', 'First Operational Date', 'Year'],
    'Puissance': ['Chip power (MW)', 'Power', 'Compute capacity (H100e)', 'Max OP/s (log)']
}

# Fonction pour trouver la bonne colonne dans le DataFrame
def trouver_colonne(noms_possibles, df_cols):
    for nom in noms_possibles:
        if nom in df_cols:
            return nom
    return None # Si aucune n'est trouvée

# 3. Filtrage dynamique
colonnes_a_garder = []
mapping_colonnes = {}

for categorie, cles in colonnes_possibles.items():
    col_trouvee = trouver_colonne(cles, df.columns)
    if col_trouvee:
        colonnes_a_garder.append(col_trouvee)
        mapping_colonnes[categorie] = col_trouvee

df_clean = df[colonnes_a_garder].copy()

# 4. Nettoyage si on a trouvé les colonnes vitales (Propriétaire et Quantité)
col_prop = mapping_colonnes.get('Proprietaire')
col_quant = mapping_colonnes.get('Quantite')

if col_prop and col_quant:
    # Supprimer les lignes vides
    df_clean = df_clean.dropna(subset=[col_prop, col_quant])
    
    # Convertir la quantité en nombre
    # On enlève les virgules éventuelles (ex: "10,000" -> 10000) et on convertit en numérique
    if df_clean[col_quant].dtype == object:
        df_clean[col_quant] = df_clean[col_quant].astype(str).str.replace(',', '', regex=False)
    
    df_clean[col_quant] = pd.to_numeric(df_clean[col_quant], errors='coerce').fillna(0).astype(int)

    # Sauvegarde du fichier propre
    fichier_sortie = "ai_chip_data_propre.csv"
    df_clean.to_csv(fichier_sortie, index=False)
    
    print(f"Nettoyage réussi ! Fichier sauvegardé sous : {fichier_sortie}")
    print("\nAperçu des données nettoyées :")
    print(df_clean.head())
else:
    print(f"Impossible de nettoyer : les colonnes correspondant au Propriétaire ou à la Quantité sont introuvables.")
    print("Vérifie la liste des colonnes disponibles affichée plus haut.")

Téléchargement des données sur les puces IA depuis Epoch AI...


KeyError: ['Cumulative units']